In [ ]:
!pip install groq -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.environ["GROQ_API_KEY"] = ""

In [ ]:

import os
import json
import numpy as np
import joblib
from scipy.spatial.distance import cdist
from groq import Groq

PROJECT_DIR = ''
EPS = 0.30
FEATURE_COLS = ['avg_amount', 'std_amount', 'max_avg_ratio', 'txn_count',
                'txn_per_active_day', 'dispersion_index']


scaler = joblib.load(f'{PROJECT_DIR}/fitted_scaler.pkl')
pca = joblib.load(f'{PROJECT_DIR}/fitted_pca.pkl')
core_points = np.load(f'{PROJECT_DIR}/core_points.npy')
clf = joblib.load(f'{PROJECT_DIR}/fraud_classifier_xgb.pkl')



client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL = "openai/gpt-oss-120b"

SYSTEM_PROMPT = """You are a fraud-review assistant for a payments platform.
You receive TWO independent signals about a recipient account, plus its raw
behavioral features:
1. An unsupervised anomaly signal (distance from known normal-pattern accounts,
   from DBSCAN clustering)
2. A supervised fraud probability (from a trained classifier)

You do NOT have access to raw transaction data or personal information.
You NEVER approve, block, or move money -- only produce a structured risk
assessment for a human reviewer. If the two signals disagree, say so explicitly
and explain what that disagreement might mean.

Respond with ONLY valid JSON, no markdown fences, no preamble:
{
  "risk_level": "low" | "medium" | "high",
  "explanation": "1-3 sentences referencing both signals and key feature values",
  "signals_agree": true | false,
  "recommended_action": "monitor" | "alert" | "escalate" | "hold_for_review",
  "confidence": "low" | "medium" | "high"
}
"""


def get_dbscan_signal(raw_features: dict) -> dict:
    x_raw = np.array([[raw_features[c] for c in FEATURE_COLS]])
    for col in ['txn_count', 'txn_per_active_day', 'dispersion_index']:
        idx = FEATURE_COLS.index(col)
        x_raw[0, idx] = np.log1p(x_raw[0, idx])
    x_scaled = scaler.transform(x_raw)
    x_pca = pca.transform(x_scaled)
    min_dist = float(cdist(x_pca, core_points).min())
    return {"is_anomalous": min_dist > EPS, "distance": min_dist}


def get_classifier_signal(raw_features: dict) -> dict:
    x = np.array([[raw_features[c] for c in FEATURE_COLS]])
    x_log = x.copy()
    for col in ['txn_count', 'txn_per_active_day', 'dispersion_index']:
        idx = FEATURE_COLS.index(col)
        x_log[0, idx] = np.log1p(x_log[0, idx])
    prob = float(clf.predict_proba(x_log)[0, 1])
    return {"fraud_probability": prob}


def get_agent_verdict(account_id: str, raw_features: dict,
                       dbscan_signal: dict, clf_signal: dict) -> dict:
    feat_lines = "\n".join(f"- {k}: {v:.4f}" for k, v in raw_features.items())
    user_prompt = f"""Account ID: {account_id}

Features:
{feat_lines}

Signal 1 (DBSCAN, unsupervised): is_anomalous={dbscan_signal['is_anomalous']}, distance_to_nearest_core={dbscan_signal['distance']:.3f} (threshold={EPS})
Signal 2 (XGBoost classifier, supervised): fraud_probability={clf_signal['fraud_probability']:.4f}

Assess this account's fraud risk using both signals."""

    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": user_prompt}],
        temperature=0.2,
        max_tokens=1000,
    )
    raw = resp.choices[0].message.content.strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    result = json.loads(raw)
    result["account_id"] = account_id
    return result


def score_account(account_id: str, raw_features: dict) -> dict:
    """Full pipeline: DBSCAN + classifier + agent, one call."""
    dbscan_signal = get_dbscan_signal(raw_features)
    clf_signal = get_classifier_signal(raw_features)
    verdict = get_agent_verdict(account_id, raw_features, dbscan_signal, clf_signal)
    verdict["dbscan_distance"] = dbscan_signal["distance"]
    verdict["classifier_probability"] = clf_signal["fraud_probability"]
    return verdict


if __name__ == "__main__":
    example = {
        'avg_amount': 85000.0, 'std_amount': 500.0, 'max_avg_ratio': 1.1,
        'txn_count': 3, 'txn_per_active_day': 3.0, 'dispersion_index': 1.2,
    }
    print(score_account("C_TEST_001", example))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


{'risk_level': 'medium', 'explanation': 'DBSCAN flags the account as anomalous (distance 1.004 > 0.3) despite the XGBoost model assigning a very low fraud probability (0.14%). The high average transaction amount (85,000) with low variance and only three transactions contributes to the unsupervised outlier detection, while the supervised model finds no fraud pattern.', 'signals_agree': False, 'recommended_action': 'hold_for_review', 'confidence': 'medium', 'account_id': 'C_TEST_001', 'dbscan_distance': 1.0037463465042487, 'classifier_probability': 0.001382835558615625}
